# ZeRO (Zero Redundancy Optimizer) Tutorial

## Overview

ZeRO is DeepSpeed's memory optimization technique that partitions optimizer states, gradients, and parameters across data parallel processes.

### Learning Objectives
- Understand ZeRO Stage 1, 2, and 3
- Analyze memory savings at each stage
- Implement ZeRO with DeepSpeed
- Compare ZeRO vs FSDP

### References
- Rajbhandari et al., "ZeRO: Memory Optimizations Toward Training Trillion Parameter Models", SC 2020
- [DeepSpeed ZeRO Documentation](https://www.deepspeed.ai/tutorials/zero/)

## 1. Mathematical Foundation

### 1.1 Memory Breakdown

For a model with $\Psi$ parameters using Adam optimizer with mixed precision:

| Component | Size | Description |
|-----------|------|-------------|
| Parameters (FP16) | $2\Psi$ | Model weights |
| Gradients (FP16) | $2\Psi$ | Gradient storage |
| Optimizer States | $12\Psi$ | Adam: FP32 params + momentum + variance |
| **Total** | $16\Psi$ | Per GPU in DDP |

### 1.2 ZeRO Stages Memory Formula

With $N$ GPUs:

**Stage 1 (Optimizer State Partitioning):**
$$M_1 = 2\Psi + 2\Psi + \frac{12\Psi}{N} = 4\Psi + \frac{12\Psi}{N}$$

**Stage 2 (+ Gradient Partitioning):**
$$M_2 = 2\Psi + \frac{2\Psi}{N} + \frac{12\Psi}{N} = 2\Psi + \frac{14\Psi}{N}$$

**Stage 3 (+ Parameter Partitioning):**
$$M_3 = \frac{2\Psi}{N} + \frac{2\Psi}{N} + \frac{12\Psi}{N} = \frac{16\Psi}{N}$$

## 2. ZeRO Architecture Visualization

```
┌─────────────────────────────────────────────────────────────────────┐
│                    ZeRO Stages Comparison                           │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  DDP (Baseline):     GPU0: [P][G][O]  GPU1: [P][G][O]  (Full copy) │
│                                                                     │
│  ZeRO Stage 1:       GPU0: [P][G][O0] GPU1: [P][G][O1] (Shard O)   │
│                                                                     │
│  ZeRO Stage 2:       GPU0: [P][G0][O0] GPU1: [P][G1][O1] (Shard G,O)│
│                                                                     │
│  ZeRO Stage 3:       GPU0: [P0][G0][O0] GPU1: [P1][G1][O1] (All)   │
│                                                                     │
│  Legend: P=Parameters, G=Gradients, O=Optimizer States             │
└─────────────────────────────────────────────────────────────────────┘
```

In [ ]:
import torch
import torch.nn as nn

def calculate_zero_memory(num_params: int, num_gpus: int):
    """Calculate memory usage for each ZeRO stage.
    
    Args:
        num_params: Number of model parameters
        num_gpus: Number of GPUs
    """
    psi = num_params
    N = num_gpus
    
    # Memory in bytes (assuming FP16 params/grads, FP32 optimizer)
    ddp = 16 * psi  # Full replication
    stage1 = 4 * psi + 12 * psi / N  # Shard optimizer
    stage2 = 2 * psi + 14 * psi / N  # Shard optimizer + gradients
    stage3 = 16 * psi / N  # Shard everything
    
    print(f"Model: {psi/1e9:.1f}B params, {N} GPUs")
    print(f"\n{'Stage':<12} {'Memory/GPU':<15} {'vs DDP':<10}")
    print("-" * 40)
    print(f"{'DDP':<12} {ddp/1e9:.2f} GB{'':<8} 1.00x")
    print(f"{'ZeRO-1':<12} {stage1/1e9:.2f} GB{'':<8} {ddp/stage1:.2f}x")
    print(f"{'ZeRO-2':<12} {stage2/1e9:.2f} GB{'':<8} {ddp/stage2:.2f}x")
    print(f"{'ZeRO-3':<12} {stage3/1e9:.2f} GB{'':<8} {ddp/stage3:.2f}x")

# Example: 7B model on 8 GPUs
calculate_zero_memory(7_000_000_000, 8)

## 3. DeepSpeed ZeRO Configuration

In [ ]:
def get_zero_config(stage: int, offload: bool = False) -> dict:
    """Generate DeepSpeed ZeRO configuration.
    
    Args:
        stage: ZeRO stage (1, 2, or 3)
        offload: Enable CPU offloading
    """
    config = {
        "zero_optimization": {
            "stage": stage,
            "allgather_partitions": True,
            "allgather_bucket_size": 2e8,
            "reduce_scatter": True,
            "reduce_bucket_size": 2e8,
            "overlap_comm": True,
            "contiguous_gradients": True,
        },
        "fp16": {
            "enabled": True,
            "loss_scale": 0,
            "initial_scale_power": 16,
        },
        "gradient_accumulation_steps": 1,
        "train_batch_size": "auto",
        "train_micro_batch_size_per_gpu": "auto",
    }
    
    if stage == 3:
        config["zero_optimization"].update({
            "stage3_prefetch_bucket_size": 5e7,
            "stage3_param_persistence_threshold": 1e4,
            "stage3_max_live_parameters": 1e9,
            "stage3_max_reuse_distance": 1e9,
        })
    
    if offload:
        config["zero_optimization"]["offload_optimizer"] = {
            "device": "cpu",
            "pin_memory": True
        }
        if stage == 3:
            config["zero_optimization"]["offload_param"] = {
                "device": "cpu",
                "pin_memory": True
            }
    
    return config

# Show Stage 2 config
import json
print("ZeRO Stage 2 Config:")
print(json.dumps(get_zero_config(2), indent=2))

## 4. ZeRO Implementation Example

In [ ]:
class SimpleTransformer(nn.Module):
    """Simple transformer for ZeRO demonstration."""
    def __init__(self, vocab_size=10000, d_model=512, num_layers=6):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model, nhead=8, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers)
        self.output = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        x = self.embedding(x)
        x = self.encoder(x)
        return self.output(x)

# Count parameters
model = SimpleTransformer()
num_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {num_params:,} ({num_params/1e6:.1f}M)")

In [ ]:
# DeepSpeed training script template
deepspeed_training_script = '''
import deepspeed
import torch
from torch.utils.data import DataLoader, DistributedSampler

def train_with_deepspeed(model, train_dataset, config_path):
    """Train model with DeepSpeed ZeRO."""
    
    # Initialize DeepSpeed
    model_engine, optimizer, _, _ = deepspeed.initialize(
        model=model,
        config=config_path,
        model_parameters=model.parameters()
    )
    
    # Create distributed dataloader
    sampler = DistributedSampler(train_dataset)
    dataloader = DataLoader(train_dataset, batch_size=8, sampler=sampler)
    
    # Training loop
    for epoch in range(num_epochs):
        sampler.set_epoch(epoch)
        
        for batch in dataloader:
            inputs, labels = batch
            
            # Forward pass
            outputs = model_engine(inputs)
            loss = criterion(outputs, labels)
            
            # Backward pass (DeepSpeed handles gradient sync)
            model_engine.backward(loss)
            
            # Optimizer step
            model_engine.step()
'''
print(deepspeed_training_script)

## 5. ZeRO vs FSDP Comparison

| Feature | ZeRO (DeepSpeed) | FSDP (PyTorch) |
|---------|------------------|----------------|
| **Stages** | 3 stages (1,2,3) | 3 strategies |
| **CPU Offload** | ZeRO-Offload | CPUOffload |
| **NVMe Offload** | ZeRO-Infinity | Not native |
| **Integration** | Separate library | Native PyTorch |
| **Activation Checkpointing** | Built-in | Separate API |
| **Mixed Precision** | Built-in | MixedPrecision |
| **Maturity** | More mature | Rapidly improving |

## 6. Summary

### Key Takeaways

1. **ZeRO-1**: Partitions optimizer states → ~4x memory reduction
2. **ZeRO-2**: + Gradient partitioning → ~8x memory reduction  
3. **ZeRO-3**: + Parameter partitioning → Linear scaling with GPUs
4. **ZeRO-Offload**: CPU/NVMe offloading for extreme memory savings

### Recommendations

| Model Size | Recommendation |
|------------|----------------|
| < 1B params | DDP or ZeRO-1 |
| 1-10B params | ZeRO-2 |
| 10-100B params | ZeRO-3 |
| > 100B params | ZeRO-3 + Offload |